In [1]:
# Google Colab / local setup
import os, sys, subprocess
from pathlib import Path
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CODE_ROOT = Path("/content/drive/MyDrive/msc project/code")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e",
        str(CODE_ROOT), 'catboost==1.2.10', 'rapidfuzz==3.14.3',
    ])
else:
    CODE_ROOT = next(
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").exists()
    )
os.chdir(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
CODE = CODE_ROOT

# 06c - DeBERTa sentiment

## Methodology

DeBERTa scores first-seen news and direct Federal Reserve or Truth Social events available before each M15 decision and produces five causal matched features. The same nine-model protocol uses 2024 OOF evidence, H1 2025 calibration and descriptive July 2025-March 2026 economics.

USA500 and USATECH are separate. Every economic table reports Net, Sharpe,
Sortino and drawdown after a fixed all-in round-trip deduction: 2 bps (0.02%)
for USA500 and 3 bps (0.03%) for USATECH. This covers bid-ask spread plus
commission/slippage; bar-specific spreads remain diagnostic, not additional
costs. Inputs stop before `2026-04-01 00:00 UTC`; Q2-2026 is evaluated separately in Notebook 07 and
no model is refitted here.

In [2]:
import json
import numpy as np
import pandas as pd
from IPython.display import display
from experiments.build_notebook_06 import MODEL_LABELS
from experiments.index_replication_protocol import MODEL_NAMES

STREAMS = ("usa500", "usatech")
ROOT = CODE_ROOT / "experiments" / "cache" / "index_replication"
FORWARD_ROOT = CODE_ROOT / "experiments" / "cache" / "index_all_model_forward"
RAW = CODE_ROOT / "sentiment" / "raw"
selected, forward = {}, {}
for stream in STREAMS:
    for prefix in ("", "direct_events_"):
        manifest = json.loads((RAW / f"scores_{prefix}{stream}.manifest.json").read_text(encoding="utf-8"))
        assert manifest["complete"] and manifest["cutoff_exclusive"].startswith("2026-04-01")
    selected[stream] = pd.read_parquet(ROOT / stream / "h1_selected_policies.parquet")
    diagnostic = json.loads((FORWARD_ROOT / stream / "result.json").read_text(encoding="utf-8"))
    assert diagnostic["q2_loaded"] is False and diagnostic["forward_rows"] == 36
    assert diagnostic["evidence_role"] == "secondary_reused_forward_diagnostic"
    forward[stream] = pd.read_parquet(FORWARD_ROOT / stream / "forward_summary.parquet")
    scope = selected[stream][selected[stream]["arm"].eq("deberta_matched")]
    assert scope["model_name"].nunique() == 9 and set(scope["model_name"]) == set(MODEL_NAMES)

## Feature dictionary

Method: The dictionary defines each matched DeBERTa feature and the information available before its M15 trading decision.

In [3]:
# non-economic-table
feature_dictionary = pd.DataFrame([
    ("sent_news_decay", "Time-decayed DeBERTa sentiment of first-seen financial headlines", "Only headlines available before the current bar"),
    ("sent_news_count_24h", "Number of first-seen matched headlines in the previous 24 hours", "Backward-looking rolling count"),
    ("sent_direct_decay", "Time-decayed sentiment from first-party Federal Reserve statements/minutes and Trump Truth Social posts", "Only direct_events available before the current bar; this is not linguistic directness"),
    ("sent_tone_decay", "Time-decayed native GDELT news tone", "Only first-seen news available before the current bar"),
    ("sent_macro_decay", "Time-decayed indicator of scheduled FRED macro releases", "Release timestamp must precede the current bar"),
], columns=["Feature", "Meaning", "Causal availability"])
display(feature_dictionary)
print("Takeaway: direct_events is a causal first-party event stream, not a measure of how directly a sentence is written.")

,Feature,Meaning,Causal availability
0,sent_news_decay,Time-decayed DeBERTa sentiment of first-seen f...,Only headlines available before the current bar
1,sent_news_count_24h,Number of first-seen matched headlines in the ...,Backward-looking rolling count
2,sent_direct_decay,Time-decayed sentiment from first-party Federa...,Only direct_events available before the curren...
3,sent_tone_decay,Time-decayed native GDELT news tone,Only first-seen news available before the curr...
4,sent_macro_decay,Time-decayed indicator of scheduled FRED macro...,Release timestamp must precede the current bar


Takeaway: direct_events is a causal first-party event stream, not a measure of how directly a sentence is written.


## H1 results for nine individual models

Method: The selected price/VIX base and matched DeBERTa features use the same January-June 2025 calibration rule across all nine model families.

In [4]:
# economic-table
rows = []
for stream in STREAMS:
    scope = selected[stream][selected[stream]["arm"].isin(["selected_base", "deberta_matched"])].copy()
    scope["Index"] = stream.upper(); rows.append(scope)
h1 = pd.concat(rows, ignore_index=True).replace({"arm": {"selected_base": "Price/VIX base", "deberta_matched": "DeBERTa matched"}}).rename(columns={
    "arm": "Features", "model_name": "Model", "width_bps": "Width bps", "tau": "Threshold",
    "eligible": "H1 status", "trades": "Trades", "n_long": "LONG", "n_short": "SHORT",
    "net_return": "Net %", "daily_sharpe": "Sharpe", "daily_sortino": "Sortino",
})
h1["Net %"] *= 100
h1["Model"] = h1["Model"].map(MODEL_LABELS)
h1["H1 status"] = h1["H1 status"].map({True: "Pass", False: "Below gate"})
h1 = h1.sort_values(["Net %", "Sortino", "Sharpe"], ascending=False)
display(h1[["Index", "Features", "Model", "Width bps", "Threshold", "H1 status", "Trades", "LONG", "SHORT", "Net %", "Sharpe", "Sortino"]].round(3))
best_h1 = h1.iloc[0]
print(f"Takeaway: {best_h1['Index']} {best_h1['Features']} {best_h1['Model']} has the highest H1 Net at {best_h1['Net %']:.2f}%.")

,Index,Features,Model,Width bps,Threshold,H1 status,Trades,LONG,SHORT,Net %,Sharpe,Sortino
27,USATECH,DeBERTa matched,LogReg,10,0.65,Below gate,160,138,22,10.519,1.187,1.760
18,USATECH,Price/VIX base,LogReg,10,0.65,Below gate,134,115,19,9.643,1.131,1.254
33,USATECH,DeBERTa matched,MLP,15,0.80,Pass,85,56,29,4.070,1.933,1.140
0,USA500,Price/VIX base,LogReg,15,0.80,Below gate,205,119,86,3.353,0.978,0.427
23,USATECH,Price/VIX base,CatBoost,15,0.75,Pass,110,85,25,2.986,0.738,0.325
9,USA500,DeBERTa matched,LogReg,5,0.60,Below gate,313,256,57,2.815,0.650,0.614
6,USA500,Price/VIX base,MLP,15,0.60,Below gate,725,571,154,2.722,0.373,0.248
21,USATECH,Price/VIX base,Linear SVM,15,0.45,Below gate,443,413,30,1.503,0.118,0.063
12,USA500,DeBERTa matched,Linear SVM,15,0.40,Pass,383,364,19,1.213,0.164,0.088
32,USATECH,DeBERTa matched,CatBoost,10,0.70,Below gate,220,181,39,0.999,0.298,0.198


Takeaway: USATECH DeBERTa matched LogReg has the highest H1 Net at 10.52%.


## Forward results

Method: Every DeBERTa model retains its H1-selected width and threshold for the same July 2025-March 2026 descriptive economic comparison.

In [5]:
# forward-all-models
# economic-table
active = pd.concat([
    forward[stream][forward[stream]["arm"].eq("deberta_matched")].assign(Index=stream.upper())
    for stream in STREAMS
], ignore_index=True).replace({"arm": {"deberta_matched": "DeBERTa matched"}}).rename(columns={
    "arm": "Features", "model_name": "Model", "trades": "Trades", "trades_per_day": "Trades/day",
    "n_long": "LONG", "n_short": "SHORT", "net_return": "Net %", "daily_sharpe": "Sharpe",
    "daily_sortino": "Sortino", "max_drawdown": "Max drawdown %",
})
active["Model"] = active["Model"].map(MODEL_LABELS)
active["H1 status"] = active["h1_eligible"].map({True: "Pass", False: "Below gate"})
active["Forward status"] = active["status"].map({"traded": "Traded", "no_trades": "No trades"})
active["Net %"] *= 100; active["Max drawdown %"] *= 100
result_columns = ["Trades", "Trades/day", "LONG", "SHORT", "Net %", "Sharpe", "Sortino", "Max drawdown %"]
assert len(active) == 18 and active.groupby("Index")["Model"].nunique().eq(9).all()
assert np.isfinite(active[result_columns].to_numpy(dtype=float)).all()
active = active.sort_values(["Net %", "Sortino", "Model"], ascending=[False, False, True])
display(active[["Index", "Features", "Model", "H1 status", "Forward status", *result_columns]].round(3))
best = active.iloc[0]
print(f"Takeaway: {best['Index']} {best['Model']} has the highest DeBERTa Net at {best['Net %']:.2f}%; this secondary forward result is descriptive.")

,Index,Features,Model,H1 status,Forward status,Trades,Trades/day,LONG,SHORT,Net %,Sharpe,Sortino,Max drawdown %
6,USA500,DeBERTa matched,MLP,Below gate,Traded,17,0.062,13,4,2.295,2.380,1.614,-0.110
11,USATECH,DeBERTa matched,Random Forest,Below gate,Traded,67,0.245,31,36,0.592,0.461,0.162,-1.139
2,USA500,DeBERTa matched,Random Forest,Below gate,Traded,98,0.358,75,23,0.336,0.170,0.046,-2.382
1,USA500,DeBERTa matched,Decision Tree,Below gate,No trades,0,0.000,0,0,0.000,0.000,0.000,0.000
9,USATECH,DeBERTa matched,LogReg,Below gate,No trades,0,0.000,0,0,0.000,0.000,0.000,0.000
15,USATECH,DeBERTa matched,MLP,Pass,No trades,0,0.000,0,0,0.000,0.000,0.000,0.000
7,USA500,DeBERTa matched,LSTM,Below gate,Traded,63,0.230,46,17,-0.110,-0.100,-0.036,-1.464
4,USA500,DeBERTa matched,XGBoost,Below gate,Traded,16,0.058,15,1,-0.168,-0.269,-0.060,-0.567
3,USA500,DeBERTa matched,Linear SVM,Pass,Traded,13,0.047,7,6,-0.298,-0.150,-0.015,-2.027
16,USATECH,DeBERTa matched,LSTM,Pass,Traded,41,0.150,37,4,-0.354,-0.354,-0.098,-1.111


Takeaway: USA500 MLP has the highest DeBERTa Net at 2.30%; this secondary forward result is descriptive.
